## Import & loading

In [1]:
import joblib
import numpy as np
import pandas as pd
from google.colab import drive

drive.mount("/content/drive")

# Load model and encoder
rf = joblib.load("/content/drive/MyDrive/SE-dataset/random_forest.pkl")
le = joblib.load("/content/drive/MyDrive/SE-dataset/label_encoder.pkl")

Mounted at /content/drive


## Retrieving exact same columns used to learn

In [2]:
SYMPTOM_COLUMNS = (
    pd.read_csv("/content/drive/MyDrive/SE-dataset/cleaned_diseases_symptoms.csv")
    .drop(columns=["diseases"])
    .columns
    .tolist()
)

joblib.dump(SYMPTOM_COLUMNS, "/content/drive/MyDrive/SE-dataset/symptom_columns.pkl")

from google.colab import files
files.download('/content/drive/MyDrive/SE-dataset/symptom_columns.pkl')

['/content/drive/MyDrive/SE-dataset/symptom_columns.pkl']

Saving all symptom columns from the csv format to ensure tests are carried out well

## PREDICT FUNCTION (CUSTOMIZABLE TO RETURN TOP N prediction)

In [3]:
def predict(symptoms: list[str], top_n: int = 5) -> list[dict]:
    """
    Takes a list of symptoms and returns top n disease predictions
    """

    input_vector = np.zeros(len(SYMPTOM_COLUMNS))

    for symptom in symptoms:
        symptom = symptom.strip().lower()

        if symptom in SYMPTOM_COLUMNS:
            idx = SYMPTOM_COLUMNS.index(symptom)
            input_vector[idx] = 1
        else:
            print(f"Warning: unknown symptom '{symptom}'")

    input_df = pd.DataFrame([input_vector], columns=SYMPTOM_COLUMNS)

    # Get probabilities
    probabilities = rf.predict_proba(input_df)[0]

    # Get top n predictions
    top_indices = np.argsort(probabilities)[::-1][:top_n]

    results = []
    for idx in top_indices:
        encoded_classes = rf.classes_[idx]
        disease = le.inverse_transform([encoded_classes])[0]

        results.append(
            {
                "disease": disease,
                "probability": round(float(probabilities[idx]) * 100, 2),
            }
        )

    return results

The function handles the full inference process. It starts by creating a binary input vector with one position for every symptom in `SYMPTOM_COLUMNS`. For each symptom provided, it cleans the text, checks whether the symptom is recognized, and sets the corresponding position to 1. Unknown symptoms are ignored with a warning.

The vector is then converted into a DataFrame using the same feature names and order the Random Forest was trained on. `predict_proba()` returns the model’s probability distribution across all disease classes. The probabilities are sorted from highest to lowest, the top n classes are decoded back into disease names using the label encoder, and the function returns the disease names with their percentage scores.

## MAIN

In [4]:
if __name__ == "__main__":
  SYMPTOM_GROUPS = {
    "respiratory": [
        "cough",
        "shortness of breath",
        "wheezing",
        "chest tightness",
        "coughing up sputum",
        "fever",
    ],

    "musculoskeletal": [
        "joint pain",
        "joint swelling",
        "muscle pain",
        "muscle weakness",
        "joint stiffness or tightness",
        "cramps and spasms",
    ],

    "neurological": [
        "headache",
        "dizziness",
        "loss of sensation",
        "focal weakness",
        "slurring words",
        "disturbance of memory",
    ],

    "gastrointestinal": [
        "nausea",
        "vomiting",
        "diarrhea",
        "sharp abdominal pain",
        "upper abdominal pain",
        "stomach bloating",
    ],

    "urinary": [
        "painful urination",
        "frequent urination",
        "blood in urine",
        "lower abdominal pain",
        "unusual color or odor to urine",
        "fever",
    ],

    "cardiac": [
        "sharp chest pain",
        "chest tightness",
        "palpitations",
        "irregular heartbeat",
        "shortness of breath",
        "dizziness",
    ],

    "skin": [
        "skin rash",
        "itching of skin",
        "skin irritation",
        "skin swelling",
        "skin pain",
        "abnormal appearing skin",
    ],

    "mental_health": [
        "anxiety and nervousness",
        "depression",
        "insomnia",
        "low self-esteem",
        "fears and phobias",
        "obsessions and compulsions",
    ],

    "eye": [
        "pain in eye",
        "eye redness",
        "diminished vision",
        "double vision",
        "itchiness of eye",
        "lacrimation",
    ],

    "ent": [                  # eyes, nose and throat
        "sore throat",
        "nasal congestion",
        "ear pain",
        "diminished hearing",
        "sinus congestion",
        "painful sinuses",
    ],
}

  # Just to confirm symptoms are actually in the csv and recognised
  for category, symptoms in SYMPTOM_GROUPS.items():
    valid = [s for s in symptoms if s in SYMPTOM_COLUMNS]
    invalid = [s for s in symptoms if s not in SYMPTOM_COLUMNS]

    print(f"\n{category.upper()}")
    print("Valid:", valid)
    print("Invalid:", invalid)

    predictions = predict(symptoms)
    for p in predictions:
      print(f"{p['disease']}: {p['probability']}%")


RESPIRATORY
Valid: ['cough', 'shortness of breath', 'wheezing', 'chest tightness', 'coughing up sputum', 'fever']
Invalid: []
asthma: 39.51%
chronic obstructive pulmonary disease (copd): 27.34%
acute bronchitis: 8.14%
acute respiratory distress syndrome (ards): 5.13%
acute bronchospasm: 3.01%

MUSCULOSKELETAL
Valid: ['joint pain', 'joint swelling', 'muscle pain', 'muscle weakness', 'joint stiffness or tightness', 'cramps and spasms']
Invalid: []
adhesive capsulitis of the shoulder: 13.06%
rheumatoid arthritis: 12.55%
plantar fasciitis: 11.44%
osteoarthritis: 9.91%
epilepsy: 9.01%

NEUROLOGICAL
Valid: ['headache', 'dizziness', 'loss of sensation', 'focal weakness', 'slurring words', 'disturbance of memory']
Invalid: []
stroke: 32.29%
brain cancer: 8.67%
transient ischemic attack: 8.08%
hemiplegia: 6.61%
encephalitis: 6.54%

GASTROINTESTINAL
Valid: ['nausea', 'vomiting', 'diarrhea', 'sharp abdominal pain', 'upper abdominal pain', 'stomach bloating']
Invalid: []
appendicitis: 27.39%
indi

The `main` section is just a testing and validation setup. `SYMPTOM_GROUPS` contains realistic groups of symptoms from different body systems, such as respiratory, neurological, urinary, and cardiac symptoms.

For each category, the code first checks that every symptom actually exists in `SYMPTOM_COLUMNS`. It prints which symptoms are valid or invalid, then passes the group into `predict()` and displays the model’s top five disease predictions.